# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: 
Date: 

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: E:\Bootcamp Repo\Bootcamp_Yunqiang_Lyu\homework\homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [8]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))

if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'TIME_SERIES_DAILY_ADJUSTED',
        'symbol': SYMBOL,
        'outputsize': 'compact',
        'apikey': os.getenv('ALPHAVANTAGE_API_KEY')
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()

    key = [k for k in js if 'Time Series' in k][0]

    df_api = (
        pd.DataFrame(js[key])
        .T
        .reset_index()
        .rename(columns={
            'index': 'date',
            '5. adjusted close': 'adj_close'
        })[['date', 'adj_close']]
    )

    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['adj_close'] = pd.to_numeric(df_api['adj_close'])

else:
    import yfinance as yf

    raw = yf.download(
        SYMBOL,
        period='3mo',
        interval='1d',
        auto_adjust=False,
        progress=False
    )

    # Handle MultiIndex columns returned by newer yfinance versions
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    df_api = (
        raw
        .reset_index()[['Date', 'Adj Close']]
        .rename(columns={
            'Date': 'date',
            'Adj Close': 'adj_close'
        })
    )

    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['adj_close'] = pd.to_numeric(
        df_api['adj_close'],
        errors='coerce'
    )

v_api = validate(
    df_api,
    ['date', 'adj_close']
)

v_api

E:\Anaconda\Anaconda1\Lib\site-packages\yfinance\scrapers\history.py:407: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


{'missing': [], 'shape': (63, 2), 'na_total': 0}

In [9]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data\raw\api_source-yfinance_symbol-AAPL_20260817-161912.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [12]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

headers = {
    'User-Agent': 'AFE-Homework/1.0'
}

try:
    resp = requests.get(
        SCRAPE_URL,
        headers=headers,
        timeout=30
    )
    resp.raise_for_status()

    soup = BeautifulSoup(
        resp.text,
        'html.parser'
    )

    table = soup.find(
        'table',
        {'id': 'constituents'}
    )

    rows = [
        [
            c.get_text(strip=True)
            for c in tr.find_all(['th', 'td'])
        ]
        for tr in table.find_all('tr')
    ]

    header, *data = [
        r for r in rows if r
    ]

    df_scrape = pd.DataFrame(
        data,
        columns=header
    )

except Exception as e:
    print("Scrape failed:", e)
    raise


print(df_scrape.head())

print("\nShape:")
print(df_scrape.shape)

print("\nColumns:")
print(df_scrape.columns.tolist())

print("\nNA counts:")
print(df_scrape.isna().sum())

v_scrape = validate(
    df_scrape,
    ['Symbol', 'Security']
)

v_scrape

  Symbol             Security              GICSSector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04   
1               Building Products     Milwaukee, Wisconsin  2017-07-26   
2           Health Care Equipment  North Chicago, Illinois  1957-03-04   
3                   Biotechnology  North Chicago, Illinois  2012-12-31   
4  IT Consulting & Other Services          Dublin, Ireland  2011-07-06   

          CIK      Founded  
0  0000066740         1902  
1  0000091142         1916  
2  0000001800         1888  
3  0001551152  2013 (1888)  
4  0001467373         1989  

Shape:
(503, 8)

Columns:
['Symbol', 

{'missing': [], 'shape': (503, 8), 'na_total': 0}

In [13]:
_ = save_csv(df_scrape, prefix='scrape', site='example', table='markets')

Saved data\raw\scrape_site-example_table-markets_20260817-162200.csv


## Documentation
- API Source: (URL/endpoint/params)
- Scrape Source: (URL/table description)
- Assumptions & risks: (rate limits, selector fragility, schema changes)
- Confirm `.env` is not committed.